# 04 · ¿Qué debe revisar una marca o un retailer?

**Objetivo:** Mostrar a revenue management y category management cómo se comporta en el anaquel la arquitectura de precios por tamaño de marcas líderes, y dónde se rompe.

**Pregunta:** ¿Cuánto paga de más el cliente en un escalón roto y qué escalones conviene revisar?

Importa porque convierte el % de rupturas en pesos: cuánto cuesta al cliente comprar el tamaño grande donde la
escalera está rota, y qué tan grande es el descuento por volumen donde sí funciona.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import escalera as S
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pts, st, cat = S.load()
REPS = 500
p = pd.read_csv(PROC / "escalones.csv")
SEL = json.loads((META / "parametros.json").read_text(encoding="utf-8"))["seleccionadas"]
x = p[p.contiguo].copy()
x["sobreprecio_unidad"] = x.precio_b - x.precio_a
x["sobreprecio_empaque"] = x.sobreprecio_unidad * x.tam_b
tab = x.groupby(["escalera", "escalon"]).agg(
    tiendas=("cambio_pct", "size"), pct_roto=("cambio_pct", lambda v: (v > S.TOL).mean() * 100),
    cambio_mediano=("cambio_pct", "median"),
    sobreprecio_empaque_si_roto=("sobreprecio_empaque", lambda v: v[v > 0].median() if (v > 0).any() else 0.0))
display(tab.round(2))

tiendas  pct_roto  cambio_mediano  sobreprecio_empaque_si_roto
escalera                   escalon                                                                          
Chiles La Costeña en rajas 105 g → 220 g           76      0.00           -9.32                         0.00
                           220 g → 240 g            1      0.00           -8.33                         0.00
                           240 g → 340 g          120     45.83           -2.80                         1.58
                           340 g → 380 g            3      0.00          -10.53                         0.00
Coca-Cola                  1 L → 2.75 L           150      0.00          -45.74                         0.00
                           355 ml → 600 ml        247      0.00          -44.46                         0.00
                           600 ml → 1 L           199      2.51          -13.91                         1.67
Huevo El Calvario          12 pzas → 18 pzas      126     10.32           -8.64                         7.90
                           18 pzas → 30 pzas      128     73.44            2.63                         3.33
Huevo San Juan             12 pzas → 18 pzas      199     85.93            2.38                         1.05
                           18 pzas → 30 pzas      191      1.57           -9.30                         9.17
Zucaritas                  30 g → 380 g           143      0.00          -51.75                         0.00
                           380 g → 600 g          137     12.41          -10.08                         5.16
                           600 g → 620 g            3      0.00           -3.23                         0.00

**Lectura.** Sobreprecio por empaque = lo que paga de más el cliente por todo el paquete grande frente a comprar
la misma cantidad al precio por unidad del chico (mediana de las tiendas donde el escalón está roto).

In [2]:
tab.to_csv(PROC / "que_revisar.csv", encoding="utf-8-sig")
F = "04_que_revisar"
t = tab.loc[("Huevo San Juan", "12 pzas → 18 pzas")]
reg("sj_sobreprecio_18", t.sobreprecio_empaque_si_roto, fuente=F, nota="pesos de más por el paquete de 18 frente al precio por pieza del de 12")
t2 = tab.loc[("Huevo El Calvario", "18 pzas → 30 pzas")]
reg("ec_sobreprecio_30", t2.sobreprecio_empaque_si_roto, fuente=F)
t3 = tab.loc[("Chiles La Costeña en rajas", "240 g → 340 g")]
reg("raj_sobreprecio_340", t3.sobreprecio_empaque_si_roto, fuente=F)
display(Markdown(
    f"**Conclusión.** Donde la escalera del huevo San Juan está rota, el paquete de 18 cuesta ${t.sobreprecio_empaque_si_roto:.2f} más que 18 "
    f"huevos al precio por pieza del de 12; en El Calvario, el de 30 cuesta ${t2.sobreprecio_empaque_si_roto:.2f} más que 30 huevos al precio "
    f"del de 18. Son pocos pesos por paquete, pero rompen la promesa de que comprar más sale más barato."))

**Conclusión.** Donde la escalera del huevo San Juan está rota, el paquete de 18 cuesta $1.05 más que 18 huevos al precio por pieza del de 12; en El Calvario, el de 30 cuesta $3.33 más que 30 huevos al precio del de 18. Son pocos pesos por paquete, pero rompen la promesa de que comprar más sale más barato.